In [ ]:
import pyspark
from pyspark import SparkContext

sc = SparkContext.getOrCreate()

data = sc.textFile("work/calidad_aire_datos_meteo_mes.csv")
cabecera = data.first()

res = (data
    .filter(lambda linea: linea != cabecera)
    .map(lambda linea: linea.replace(",", "."))
    .map(lambda linea: linea.split(";")) 
    .filter(lambda campos: campos[0] == "28")
    .filter(lambda campos: campos[3] == "83")

    .flatMap(lambda campos: [
        ((campos[7].zfill(2) + "-" + campos[6].zfill(2) + "-" + campos[5]), #este seria el elemento a que almacena la fecha, los .zfill lo que hacen es rellenar con ceros a la izquierda hasta llegar a una longitud de 2
         float(campos[i]))
        for i in range(8,56,2)
        if campos[i+1] == "V" and campos[i] != ""
    ])

    .reduceByKey(lambda a, b: max(a,b))
    .sortByKey()
)

for r in res.collect():
    print(f"{r}")
